In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# ==========================================
# 1. DATA PREPROCESSING & FEATURE ENGINEERING
# ==========================================
print("Loading data...")
df = pd.read_csv('HomeC.csv', low_memory=False)

# Clean Time and Sort (Crucial for Time-Series Lags)
df['time'] = pd.to_numeric(df['time'], errors='coerce')
df = df.dropna(subset=['time'])
df['time'] = pd.to_datetime(df['time'], unit='s')
df = df.sort_values('time') # Must be chronological for lags

# Standard Time Features
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['month'] = df['time'].dt.month
df = pd.get_dummies(df, columns=['icon', 'summary'], drop_first=True)
df = df.apply(pd.to_numeric, errors='coerce')

print("Engineering Safe Time-Series Lag Features...")
# Create Lag Features (Usage from 1 minute ago)
df['Total_Usage_Lag1'] = df['use [kW]'].shift(1)
df['Fridge_Lag1'] = df['Fridge [kW]'].shift(1)
df['Furnace_Lag1'] = df['Furnace 1 [kW]'].shift(1)

# Drop rows with NaN values caused by shifting
df = df.dropna()

# Drop the CURRENT appliance columns to completely eliminate data leakage
leakage_cols = ['time', 'House overall [kW]', 'Dishwasher [kW]', 'Furnace 1 [kW]', 
                'Furnace 2 [kW]', 'Home office [kW]', 'Fridge [kW]', 'Wine cellar [kW]', 
                'Garage door [kW]', 'Kitchen 12 [kW]', 'Kitchen 14 [kW]', 'Kitchen 38 [kW]', 
                'Barn [kW]', 'Well [kW]', 'Microwave [kW]', 'Living room [kW]', 'Solar [kW]']

X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
X = X.drop(columns=['use [kW]']) # Drop target from features
y = df['use [kW]']

# Clean feature names for XGBoost
X.columns = X.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace('<', '', regex=False)

# Split data sequentially to respect time-series flow
split_idx = int(len(df) * 0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Scaling data...")
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# ==========================================
# 2. MODEL TRAINING
# ==========================================
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, 
    tree_method='hist', n_jobs=-1, early_stopping_rounds=20
)
xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)

# ==========================================
# 3. EVALUATION & METRICS TABLE
# ==========================================
print("\nGenerating evaluation metrics...")

# Helper function to generate all 4 metrics
def get_full_metrics(y_true, y_pred, model_name, dataset_type):
    return {
        "Dataset": dataset_type,
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

# Generate predictions for both Train and Test sets
rf_train_preds = rf_model.predict(X_train_scaled)
xgb_train_preds = xgb_model.predict(X_train_scaled)
rf_test_preds = rf_model.predict(X_test_scaled)
xgb_test_preds = xgb_model.predict(X_test_scaled)

# Compile results into a list (YOUR MODELS ONLY)
metrics_list = [
    # Training Metrics
    get_full_metrics(y_train, rf_train_preds, "1. My Lagged RF", "1_Training"),
    get_full_metrics(y_train, xgb_train_preds, "2. My Lagged XGBoost", "1_Training"),
    
    # Testing Metrics
    get_full_metrics(y_test, rf_test_preds, "1. My Lagged RF", "2_Testing"),
    get_full_metrics(y_test, xgb_test_preds, "2. My Lagged XGBoost", "2_Testing")
]

# Create DataFrame and format it cleanly
metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df.set_index(['Dataset', 'Model']).sort_index()

print("\n--- Final Model Metric Table ---")
display(metrics_df.round(4))

# ==========================================
# 4. SAVE ASSETS FOR DASHBOARD
# ==========================================
print("\nSaving assets to 'models/' directory...")
os.makedirs('models', exist_ok=True)
joblib.dump(rf_model, 'models/rf_model.pkl')
joblib.dump(xgb_model, 'models/xgb_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(list(X_train.columns), 'models/feature_names.pkl')
print("Complete!")